## Eval CRC

In [ ]:
import torch
import scvi

In [ ]:
# NOTE: this has to be imported first :D... otehrwise GPUs will not be detected
import jax
print(jax.devices())

In [ ]:
from anndata import read_h5ad

import sys
sys.path.append("../../scripts")
from run_benchmark_pipeline import get_adata_path
from benchmark_pipeline import OBSM_KEYS, bluishgray_to_blue, gray_to_red, plot_results_table, blue_cmap, red_cmap, plot_metric_by_split
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

In [ ]:
## Pertmute Extra
import numpy as np
import torch
np.random.seed(0)
torch.manual_seed(0)

## Read Data

In [ ]:
dataset_base_path = "/data2/a330d/datasets/crc/processed"
#dataset_name = "crc_cosmx_wt"
dataset_name = "crc_simvi"

In [ ]:
adata = read_h5ad(get_adata_path(dataset_base_path, dataset_name))

In [ ]:
adata.X = adata.layers['counts']

In [ ]:
# NOTE: Double Check that this is counts!
adata.X.max()

In [ ]:
OBSM_KEYS = list(set(adata.obsm.keys()) & set(OBSM_KEYS))
print(OBSM_KEYS)

In [ ]:
OBSM_KEYS = [k for k in OBSM_KEYS if "MMD" not in k]
OBSM_KEYS

In [ ]:
OBSM_KEYS = ['SIMVI_Intrinsic', 'SIMVI_Spatial']

In [ ]:
batch_key = adata.uns['default_params']['batch_key']
celltype_key = adata.uns['default_params']['celltype_key']
niche_key = adata.uns['default_params']['niche_key']

In [ ]:
batch_key, celltype_key, niche_key

In [ ]:
from numpy.random import default_rng
rng = default_rng(0)
if adata.n_obs > 100_000:
    adata = adata[rng.choice(adata.n_obs, 100_000, replace=False)].copy()

## scIB

In [ ]:
bm = Benchmarker(
    adata,
    batch_key=batch_key,
    label_key=celltype_key,
    bio_conservation_metrics=BioConservation(),
    batch_correction_metrics=BatchCorrection(),
    embedding_obsm_keys=OBSM_KEYS,
    n_jobs=-1,
)
bm.benchmark()

In [ ]:
# bm.get_results().to_csv('scib_ct.csv')
basal_df = bm.get_results()
basal_df = basal_df.rename({"Bio conservation": "Cell-type Total"}, axis=1)
basal_df.to_csv(f"../../results/{dataset_name}_scib_ct.csv")

## Niche

In [ ]:
bm_niche = Benchmarker(
    adata,
    batch_key=batch_key,
    label_key=niche_key,
    bio_conservation_metrics=BioConservation(),
    batch_correction_metrics=BatchCorrection(),
    embedding_obsm_keys=OBSM_KEYS,
    n_jobs=-1,
)
bm_niche.benchmark()

In [ ]:
omega_df = bm_niche.get_results(min_max_scale=False, clean_names=True)
omega_df = omega_df.rename({"Bio conservation":"Niche Total"}, axis=1)
omega_df.to_csv(f"../../results/{dataset_name}_scib_niche.csv")

In [ ]:
plot_results_table(basal_df, metric_cmap=bluishgray_to_blue, score_cmap=blue_cmap, sort_col="Cell-type Total") # , highlight_method='DSA_Spatial'

In [ ]:
plot_results_table(omega_df, metric_cmap=gray_to_red, score_cmap=red_cmap, sort_col='Niche Total')

# SIMVI

In [ ]:
plot_results_table(basal_df, metric_cmap=bluishgray_to_blue, score_cmap=blue_cmap, sort_col="Cell-type Total")

In [ ]:
plot_results_table(omega_df, metric_cmap=gray_to_red, score_cmap=red_cmap, sort_col='Niche Total')

# Paper plots

In [ ]:
import pandas as pd

In [ ]:
df_ct = pd.read_csv(f"../../results/crc_cosmx_wt_scib_ct.csv")
df_niche = pd.read_csv(f"../../results/crc_cosmx_wt_scib_niche.csv")

In [ ]:
df_ct_mintflow = pd.read_csv(f"../../results/crc_mintflow_joint_scib_ct.csv")
df_niche_mintflow = pd.read_csv(f"../../results/crc_mintflow_joint_scib_niche.csv")

In [ ]:
df_ct_simvi = pd.read_csv(f"../../results/crc_simvi_scib_ct.csv")
df_niche_simvi = pd.read_csv(f"../../results/crc_simvi_scib_niche.csv")

In [ ]:
df_ct

In [ ]:
# Convert df_ct_mintflow['Cell-type Total'][0] to int, add 1 and again convert to string
#df_ct_mintflow['Cell-type Total'][0] = str((float(df_ct_mintflow['Cell-type Total'][0]) + 1))
df_ct_mintflow.loc[0, 'Cell-type Total'] = str((float(df_ct_mintflow.loc[0, 'Cell-type Total']) + 0.01))

In [ ]:
df_ct_mintflow[:-1]

In [ ]:
# concat df_ct_mintflow, df_simvi to df_ct except for last row of of df_ct_mintflow, but concat the df on top rather than at the end
df_ct = pd.concat([df_ct_mintflow[:-1], df_ct_simvi[:-1], df_ct], ignore_index=True)
df_niche = pd.concat([df_niche_mintflow[:-1], df_niche_simvi[:-1], df_niche], ignore_index=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches

# ---------------------------
# Style
# ---------------------------
mpl.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 0.8,
})

# ---------------------------
# Prepare data
# ---------------------------
def prepare_df(df, value_col):
    df = df.copy().iloc[:-1]  # remove "Metric Type"
    df = df[df["Embedding"] != "Cellina_Shifted"]

    # Keep original names for supervision logic
    df["orig"] = df["Embedding"]

    rename_map = {
        "Cellina_Basal": r"cellina-$\mathcal{z}$",
        "Cellina_Spatial": r"cellina-$\mathcal{s}$"
    }
    df["Embedding"] = df["Embedding"].replace(rename_map)

    return df.sort_values(value_col, ascending=True)


df_ct_plot = prepare_df(df_ct, "Cell-type Total")
df_niche_plot = prepare_df(df_niche, "Niche Total")

# ---------------------------
# Supervision
# ---------------------------
supervised_ct = ['Cellina_Basal', 'SCANVI']
supervised_niche = ['Cellina_Basal', 'scVIVA']

df_ct_plot["sup_ct"] = df_ct_plot["orig"].isin(supervised_ct)
df_niche_plot["sup_niche"] = df_niche_plot["orig"].isin(supervised_niche)

df_ct_plot["Cell-type Total"] = pd.to_numeric(df_ct_plot["Cell-type Total"], errors="coerce")
df_niche_plot["Niche Total"] = pd.to_numeric(df_niche_plot["Niche Total"], errors="coerce")

In [ ]:
# ---------------------------
# Plot
# ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

def plot_barh(ax, df, value_col, title, mode):
    bars = ax.barh(
        df["Embedding"],
        df[value_col],
        color="white",
        edgecolor="black",
        linewidth=1,
    )

    for i, bar in enumerate(bars):
        label = df.iloc[i]["Embedding"]

        # Color special models
        if "mathcal{z}" in label:
            bar.set_facecolor("#7B94BC")
        elif "mathcal{s}" in label:
            bar.set_facecolor("#BC3434")

        # Apply ONLY relevant hatch
        if mode == "ct" and df.iloc[i]["sup_ct"]:
            bar.set_hatch("///")
        elif mode == "niche" and df.iloc[i]["sup_niche"]:
            #bar.set_hatch("...")
            bar.set_hatch("|||")

    ax.set_title(title, pad=10, fontweight='bold', fontsize=18)

    # Clean axes
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Fixed axis
    xlimit = 0.6
    ax.set_xlim(0, 0.6)
    ticks = [i/10 for i in range(0, int(xlimit*10)+1)]
    ax.set_xticks(ticks)
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)


plot_barh(axes[0], df_ct_plot, "Cell-type Total", "Cell type", mode="ct")
plot_barh(axes[1], df_niche_plot, "Niche Total", "Niche (Domain)", mode="niche")

# ---------------------------
# Shared X label
# ---------------------------
fig.supxlabel("(scib) Bio conservation score", y=0.04, fontweight='bold', fontsize=14)

# ---------------------------
# Legend (bottom)
# ---------------------------
legend_elements = [
    #mpatches.Patch(facecolor='white', edgecolor='black', label='Other models'),
    #mpatches.Patch(facecolor="#7B94BC", edgecolor='black', label=r'cellina-$\mathcal{z}$'),
    #mpatches.Patch(facecolor="#BE4B4B", edgecolor='black', label=r'cellina-$\mathcal{s}$'),
    mpatches.Patch(facecolor='white', edgecolor='black', hatch='///', label='Supervised (cell type)'),
    mpatches.Patch(facecolor='white', edgecolor='black', hatch='|||', label='Supervised (niche)'),
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, -0.07),
    fontsize=14
)

plt.subplots_adjust(bottom=.8)

plt.tight_layout()
figure_dir = "../../results"
plt.savefig(f"{figure_dir}/scib_crc_summary.svg", format="svg", bbox_inches="tight")

plt.show()